#### `What is my total spending on food this month?`

In [4]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

use of a package called aisuite [aisuite repository](https://github.com/andrewyng/aisuite)  that makes it easy to call LLMs hosted by different model providers

In [5]:
import aisuite as ai

client = ai.Client()

#### Create Random personal finance dataset or you can also add your finance Details 


Using get_schema function inspect table of DB. 

In [6]:
from utils.utils import print_html, get_schema 

print_html(get_schema('personal_finance.db'))

#### Here we use llm to Query a Database. This function helps us to do that

In [49]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

In [ ]:
## schema of database I generated manually we cannot have function which can give us this type of response.

schema = """
table name: transactions
id (INTEGER)
date (DATE)(yyyy-mm-dd)
category (VARCHAR(30)) ["Food",
    "Transport",
    "Shopping",
    "Entertainment",
    "Bills",
    "Salary",
    "Freelance",
    "Health"]
type (VARCHAR(30))  ["Income", "Expense"]
amount (REAL) 
description (VARCHAR(100))
payment_method (VARCHAR(30))  [
    "UPI",
    "Cash",
    "Credit Card",
    "Debit Card",
    "Bank Transfer"
]
"""

question = "What is my savings rate (income minus expenses) for each month?"

prompt = f"""
You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

Schema: 
{schema}

user Question:
{question}

Respond with the SQL only.
"""


model = "openai:gpt-4o-mini"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
        )


sql_1 = response.choices[0].message.content.strip()


In [ ]:
## Generate SQL query to get data  - V1
sql_1 = generate_sql(question,schema,model)

In [54]:
print_html(sql_1, title="SQL Query V1")

###  Query Execution using execute_sql()

--?> see this result and find out wheather it is correct or not

In [55]:
from utils.utils import execute_sql

result = execute_sql(sql_1,'personal_finance.db')
print_html(result, title="Output of SQL Query V1 - ❌ Does NOT fully answer the question ?")

month,total_income,total_expenses,savings_rate
2026-01,464534.18,87342.64,377191.54
2026-02,132853.46,62055.83,70797.63
2026-03,124287.73,70115.57,54172.16
2026-04,253054.47,46459.54,206594.93
2026-05,124636.27,75594.42,49041.85
2026-06,183103.72,57920.69,125183.03


### Improving SQL Queries with Reflection  

In [75]:
def refine_sql(question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:

  prompt = f"""
  You are a SQL reviewer and refiner.

  User asked:
  {question}

  Original SQL:
  {sql_query}

  SQL Output:
  {df_feedback.to_markdown(index=False)}

  Table Schema:
  {schema}

  Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
  Step 2: If improvement is needed, provide a refined SQL query for SQLite.
  If the original SQL is already correct, return it unchanged.

  Return STRICT JSON with two fields:
  {{
    "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
    "refined_sql": "<final SQL to run>"
  }}
  """
      
  response = client.chat.completions.create(
      model=model,
      messages=[{"role": "user", "content": prompt}],
      temperature=0,
  )
  content = response.choices[0].message.content

  try:
      obj = json.loads(content)
      feedback = str(obj.get("feedback", "")).strip()
      refined_sql = str(obj.get("refined_sql", sql_query)).strip()
      if not refined_sql:
          refined_sql = sql_query
  except Exception:
      # Fallback if model doesn't return valid JSON
      feedback = content.strip()
      refined_sql = sql_query

  return feedback, refined_sql

In [57]:
feedback, refined_sql = refine_sql(question, sql_1, schema, model)

In [58]:
print_html(feedback, title="Feedback on V1")
print_html(refined_sql, title="Refined SQL Query (V2)")

#### Execute and display V2 results

In [ ]:
df_sql_V2 = execute_sql(refined_sql, db_path='personal_finance.db')
print_html(df_sql_V2, title="SQL Output of V2 - ✅ Fully answers the question")

month,total_income,total_expenses,savings_amount,savings_rate_percentage
2026-01,464534.18,87342.64,377191.54,81.197801
2026-02,132853.46,62055.83,70797.63,53.290016
2026-03,124287.73,70115.57,54172.16,43.586089
2026-04,253054.47,46459.54,206594.93,81.640498
2026-05,124636.27,75594.42,49041.85,39.347976
2026-06,183103.72,57920.69,125183.03,68.367278


## Database Query Workflow - Putting all things togather

In [ ]:
db_path = "personal_finance.db"
question = "What is my total spending on food this month?"

In [ ]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4o-mini",
    model_evaluation: str = "openai:gpt-4.1",
):
    """
    End-to-end workflow to generate, execute, evaluate, and refine SQL queries.

    Steps:
      1) Extract database schema
      2) Generate SQL (V1)
      3) Execute V1 → show output
      4) Reflect on V1 with execution feedback → propose refined SQL (V2)
      5) Execute V2 → show final answer
    """
    ## imports
    from utils.utils import print_html, get_schema 



    schema = """
    Database: Personal Finance

    Table: transactions

    Columns:
    - id (INTEGER, Primary Key)

    - date (DATE)
    Format: YYYY-MM-DD
    Example: 2026-06-18

    - category (VARCHAR(30))
    Allowed Values:
        - Food
        - Transport
        - Shopping
        - Entertainment
        - Bills
        - Salary
        - Freelance
        - Health

    - type (VARCHAR(30))
    Allowed Values:
        - Income
        - Expense

    - amount (REAL)
    Description: Transaction amount. Always stored as a positive number.

    - description (VARCHAR(100))
    Description: Optional note or description for the transaction.

    - payment_method (VARCHAR(30))
    Allowed Values:
        - UPI
        - Cash
        - Credit Card
        - Debit Card
        - Bank Transfer

    Business Rules:
    - Salary and Freelance are typically Income transactions.
    - Food, Transport, Shopping, Entertainment, Bills, and Health are typically Expense transactions.
    - Dates are stored in YYYY-MM-DD format.
    - Use SQLite-compatible SQL syntax.
    - Category names and type values are case-sensitive and must match the allowed values exactly.
    """

    # 1) Schema-  here we need to give this schema manually 
    # schema = get_schema(db_path)
    print_html(schema, title="📘 Step 1 — Manually Add  or Extract Database Schema")

    # 2) Generate SQL (V1)
    sql_v1 = generate_sql(question, schema, model_generation)
    print_html(
        sql_v1,
        title="🧠 Step 2 — Generate SQL (V1)"
    )

    # 3) Execute V1
    df_v1 = execute_sql(sql_v1, db_path)
    print_html(
        df_v1,
        title="🧪 Step 3 — Execute V1 (SQL Output)"
    )

    # 4) Reflect on V1 with execution feedback → refine to V2
    feedback, sql_v2 = refine_sql(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,          # external feedback: real output of V1
        schema=schema,
        model=model_evaluation,
    )
    print_html(
        feedback,
        title="🧭 Step 4 — Reflect on V1 (Feedback)"
    )
    print_html(
        sql_v2,
        title="🔁 Step 4 — Refined SQL (V2)"
    )

    # 5) Execute V2
    df_v2 = execute_sql(sql_v2, db_path)
    print_html(
        df_v2,
        title="✅ Step 5 — Execute V2 (Final Answer)"
    )



### 3.4. Run the SQL Workflow

Now it’s time for **you** to execute the complete SQL processing pipeline. You can try different combinations of the following OpenAI models, each with different capabilities and performance:

* `openai:gpt-4o`
* `openai:gpt-4.1`
* `openai:gpt-4.1-mini`
* `openai:gpt-3.5-turbo`

💡 In this workflow, `openai:gpt-4.1` often gives the best results for self-reflection tasks.

**Important:** Because Large Language Models (LLMs) are stochastic, every run may return slightly different results.
You are encouraged to experiment with different models and combinations to find the setup that works best for **you**.

In [81]:
run_sql_workflow(
    db_path,
    "What is my savings rate (income minus expenses) for each month?")

month,total_income,total_expenses,savings_rate
2026-01,464534.18,87342.64,377191.54
2026-02,132853.46,62055.83,70797.63
2026-03,124287.73,70115.57,54172.16
2026-04,253054.47,46459.54,206594.93
2026-05,124636.27,75594.42,49041.85
2026-06,183103.72,57920.69,125183.03


month,total_income,total_expenses,savings_rate
2026-01,464534.18,87342.64,377191.54
2026-02,132853.46,62055.83,70797.63
2026-03,124287.73,70115.57,54172.16
2026-04,253054.47,46459.54,206594.93
2026-05,124636.27,75594.42,49041.85
2026-06,183103.72,57920.69,125183.03
